# nnU-Net — Segmentación de Vértebras (Baseline Externo)

Segmentación semántica multiclase de vértebras en radiografías de columna.  
**Rol:** Baseline de referencia externo. No se interviene en arquitectura, loss, LR ni estrategia de entrenamiento.  
**Clases:** 23 (background + C3–C7 + T1–T12 + L1–L5)  
**No aplica:** AP@50 / mAP (segmentación semántica, no por instancia).  
**Dependencia extra:** `pip install nnunetv2 nibabel`

In [ ]:
import os
import json
import shutil
import subprocess
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import nibabel as nib
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
# === CONFIGURACIÓN ===

DATASET_ROOT    = '../MaIA_Scoliosis_Dataset'
DATASET_INDEX   = os.path.join(DATASET_ROOT, 'dataset_index.csv')
MODELS_DIR      = '../models'

# Rutas nnU-Net (variables de entorno requeridas por el framework)
NNUNET_RAW         = os.path.abspath('nnunet_data/raw')
NNUNET_PREPROCESSED = os.path.abspath('nnunet_data/preprocessed')
NNUNET_RESULTS     = os.path.abspath('nnunet_data/results')
PREDICTIONS_DIR    = os.path.abspath('nnunet_data/predictions')

DATASET_ID   = 1
DATASET_NAME = 'Spine'
NNUNET_CONFIG = '2d'
FOLD          = 0
SEED          = 42
NUM_CLASSES   = 23

CLASS_NAMES = {
    1: 'C7',  2: 'C6',  3: 'C5',  4: 'C4',  5: 'C3',
    6: 'T1',  7: 'T2',  8: 'T3',  9: 'T4',  10: 'T5',
    11: 'T6', 12: 'T7', 13: 'T8', 14: 'T9', 15: 'T10',
    16: 'T11', 17: 'T12',
    18: 'L1', 19: 'L2', 20: 'L3', 21: 'L4', 22: 'L5',
}

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
print(f'nnUNet_raw:          {NNUNET_RAW}')
print(f'nnUNet_preprocessed: {NNUNET_PREPROCESSED}')
print(f'nnUNet_results:      {NNUNET_RESULTS}')

---
## Sección 1 — Preprocesamiento

Solo se aplican los pasos 1–3 del pipeline general (grayscale, map_entity_ids, ROI crop).  
**No se aplica:** resize, CLAHE, augmentation ni normalización — nnU-Net los gestiona internamente.

### Funciones de carga

In [ ]:
def load_image(image_path: str) -> np.ndarray:
    """Carga imagen RGB uint8 desde disco."""
    return np.array(Image.open(image_path).convert('RGB'))


def load_mask(mask_path: str) -> np.ndarray:
    """Carga máscara 16-bit uint16 sin truncar IDs de clase."""
    return cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)


def load_binary_mask(binary_mask_path: str) -> np.ndarray:
    """Carga máscara binaria y la binariza para neutralizar artefactos JPEG."""
    raw = cv2.imread(binary_mask_path, cv2.IMREAD_GRAYSCALE)
    return (raw > 127).astype(np.uint8)


def load_dataset_index(csv_path: str) -> pd.DataFrame:
    """Carga dataset_index.csv como fuente canónica de rutas."""
    return pd.read_csv(csv_path)

### Transformaciones base

In [ ]:
def to_grayscale(image: np.ndarray) -> np.ndarray:
    """Conversión perceptual: L = 0.299R + 0.587G + 0.114B."""
    return (0.299 * image[:, :, 0]
            + 0.587 * image[:, :, 1]
            + 0.114 * image[:, :, 2]).astype(np.uint8)


def map_entity_ids(mask: np.ndarray) -> np.ndarray:
    """Mapea IDs 23–35 (Entity X) a 0 (background). Devuelve uint8."""
    result = mask.copy()
    result[result > 22] = 0
    return result.astype(np.uint8)


def compute_roi(binary_mask: np.ndarray, margin: float = 0.10) -> tuple:
    """Bounding box de la columna con margen ≥10%."""
    rows = np.any(binary_mask, axis=1)
    cols = np.any(binary_mask, axis=0)
    if not rows.any():
        h, w = binary_mask.shape
        return (0, 0, w, h)
    y1, y2 = np.where(rows)[0][[0, -1]]
    x1, x2 = np.where(cols)[0][[0, -1]]
    h, w = binary_mask.shape
    dy = max(1, int((y2 - y1) * margin))
    dx = max(1, int((x2 - x1) * margin))
    return (max(0, x1 - dx), max(0, y1 - dy), min(w, x2 + dx), min(h, y2 + dy))


def crop_to_roi(image: np.ndarray, mask: np.ndarray, roi: tuple) -> tuple:
    """Aplica el mismo crop a imagen y máscara."""
    x1, y1, x2, y2 = roi
    return image[y1:y2, x1:x2], mask[y1:y2, x1:x2]

### Split del dataset

In [ ]:
def split_dataset(df: pd.DataFrame, train: float = 0.70,
                  val: float = 0.15, seed: int = 42) -> tuple:
    """
    Split 70/15/15 estratificado por columna 'split' (Normal/Scoliosis).
    Mismo split que los notebooks Mask R-CNN para comparación honesta entre modelos.
    """
    train_df, temp_df = train_test_split(
        df, train_size=train, stratify=df['split'], random_state=seed
    )
    val_ratio = val / (1.0 - train)
    val_df, test_df = train_test_split(
        temp_df, train_size=val_ratio, stratify=temp_df['split'], random_state=seed
    )
    return (train_df.reset_index(drop=True),
            val_df.reset_index(drop=True),
            test_df.reset_index(drop=True))

### Conversión al formato nnU-Net v2

In [ ]:
def image_to_nifti(image: np.ndarray) -> nib.Nifti1Image:
    """Convierte imagen uint8 (H,W) a NIfTI float32 con affine identidad."""
    return nib.Nifti1Image(image.astype(np.float32), affine=np.eye(4))


def mask_to_nifti(mask: np.ndarray) -> nib.Nifti1Image:
    """Convierte máscara uint8 (H,W) con IDs 0–22 a NIfTI para nnU-Net."""
    return nib.Nifti1Image(mask.astype(np.uint8), affine=np.eye(4))


def save_nifti(nifti_img: nib.Nifti1Image, path: str) -> None:
    """Guarda imagen NIfTI comprimida (.nii.gz) en disco."""
    nib.save(nifti_img, path)


def create_nnunet_folder_structure(raw_path: str, dataset_id: int,
                                   dataset_name: str) -> dict:
    """
    Crea la estructura de carpetas requerida por nnU-Net v2:
      raw_path/
      └── Dataset{id:03d}_{name}/
          ├── imagesTr/
          ├── labelsTr/
          └── imagesTs/
    Retorna dict con los paths creados.
    """
    dataset_folder = os.path.join(raw_path, f'Dataset{dataset_id:03d}_{dataset_name}')
    paths = {
        'root':      dataset_folder,
        'imagesTr':  os.path.join(dataset_folder, 'imagesTr'),
        'labelsTr':  os.path.join(dataset_folder, 'labelsTr'),
        'imagesTs':  os.path.join(dataset_folder, 'imagesTs'),
    }
    for path in paths.values():
        os.makedirs(path, exist_ok=True)
    return paths


def generate_dataset_json(output_path: str, num_training: int) -> None:
    """
    Genera dataset.json con la metadata requerida por nnU-Net v2.
    Modalidad: X-Ray (canal único).
    Labels: background (0) + 22 vértebras (1–22).
    """
    labels = {'background': 0}
    labels.update({name: cid for cid, name in CLASS_NAMES.items()})

    dataset_json = {
        'channel_names': {'0': 'X-Ray'},
        'labels':        labels,
        'numTraining':   num_training,
        'file_ending':   '.nii.gz',
    }
    with open(output_path, 'w') as f:
        json.dump(dataset_json, f, indent=2)
    print(f'dataset.json generado en: {output_path}')


def _get_case_id(row) -> str:
    """Deriva el case_id del nombre de imagen sin extensión."""
    return os.path.splitext(row['image'])[0]


def _preprocess_single(row, dataset_root: str) -> tuple:
    """
    Aplica grayscale + map_entity_ids + ROI crop a una imagen y su máscara.
    Retorna (image_gray_cropped, mask_cropped).
    """
    image       = load_image(os.path.join(dataset_root, row['radiograph_path']))
    mask        = load_mask(os.path.join(dataset_root, row['multiclass_id_png']))
    binary_mask = load_binary_mask(os.path.join(dataset_root, row['label_binary_path']))
    image_gray  = to_grayscale(image)
    mask_mapped = map_entity_ids(mask)
    roi         = compute_roi(binary_mask)
    image_crop, mask_crop = crop_to_roi(image_gray, mask_mapped, roi)
    return image_crop, mask_crop


def convert_dataset(train_df: pd.DataFrame, val_df: pd.DataFrame,
                    test_df: pd.DataFrame, paths: dict,
                    dataset_root: str = DATASET_ROOT) -> None:
    """
    Convierte el dataset MaIA al formato nnU-Net v2:
      - train_df + val_df → imagesTr/ y labelsTr/
        (nnU-Net hace su propia CV interna sobre estos datos)
      - test_df → imagesTs/ (sin labels; se usa para predicción final)
    Nombrado de archivos:
      - Imágenes: {case_id}_0000.nii.gz  (sufijo _0000 = modalidad 0)
      - Máscaras: {case_id}.nii.gz
    """
    # Train + Val → imagesTr / labelsTr
    trainval_df = pd.concat([train_df, val_df], ignore_index=True)
    for _, row in trainval_df.iterrows():
        case_id    = _get_case_id(row)
        img, mask  = _preprocess_single(row, dataset_root)
        save_nifti(image_to_nifti(img),
                   os.path.join(paths['imagesTr'], f'{case_id}_0000.nii.gz'))
        save_nifti(mask_to_nifti(mask),
                   os.path.join(paths['labelsTr'], f'{case_id}.nii.gz'))

    # Test → imagesTs (solo imágenes, sin labels)
    for _, row in test_df.iterrows():
        case_id   = _get_case_id(row)
        img, _    = _preprocess_single(row, dataset_root)
        save_nifti(image_to_nifti(img),
                   os.path.join(paths['imagesTs'], f'{case_id}_0000.nii.gz'))

    print(f'Convertido: {len(trainval_df)} imágenes en imagesTr, '
          f'{len(test_df)} imágenes en imagesTs')

---
## Sección 2 — Procesamiento (Entrenamiento nnU-Net)

### Variables de entorno y comandos CLI

In [ ]:
def set_nnunet_env_vars(raw_path: str, preprocessed_path: str,
                        results_path: str) -> None:
    """Configura las variables de entorno requeridas por nnU-Net v2 en la sesión actual."""
    os.environ['nnUNet_raw']          = raw_path
    os.environ['nnUNet_preprocessed'] = preprocessed_path
    os.environ['nnUNet_results']      = results_path
    os.makedirs(raw_path, exist_ok=True)
    os.makedirs(preprocessed_path, exist_ok=True)
    os.makedirs(results_path, exist_ok=True)
    print('Variables de entorno nnU-Net configuradas.')


def _run_cmd(cmd: list) -> None:
    """Ejecuta un comando CLI, muestra salida en tiempo real y falla si hay error."""
    print(f'\n$ {" ".join(cmd)}')
    result = subprocess.run(cmd, check=True, text=True)


def run_fingerprint_extraction(dataset_id: int = DATASET_ID) -> None:
    """
    Extrae el fingerprint del dataset.
    nnU-Net analiza estadísticas del dataset para autoconfigurar el plan de entrenamiento.
    """
    _run_cmd(['nnUNetv2_extract_fingerprint', '-d', str(dataset_id), '--verify_dataset_integrity'])


def run_plan_and_preprocess(dataset_id: int = DATASET_ID,
                             config: str = NNUNET_CONFIG) -> None:
    """
    Genera el plan de entrenamiento y preprocesa los datos.
    config='2d' fuerza configuración 2D para radiografías.
    """
    _run_cmd([
        'nnUNetv2_plan_and_preprocess',
        '-d', str(dataset_id),
        '-c', config,
        '--verify_dataset_integrity',
    ])


def run_training(dataset_id: int = DATASET_ID,
                 config: str = NNUNET_CONFIG,
                 fold: int = FOLD) -> None:
    """
    Lanza el entrenamiento nnU-Net.
    Se entrena solo fold=0 (1 de los 5 folds por defecto) para reducir tiempo de cómputo.
    nnU-Net gestiona internamente: arquitectura, loss, LR, epochs, checkpointing.
    """
    _run_cmd([
        'nnUNetv2_train',
        str(dataset_id), config, str(fold),
        '--npz',
    ])


def run_predict(input_folder: str, output_folder: str,
                dataset_id: int = DATASET_ID,
                config: str = NNUNET_CONFIG,
                fold: int = FOLD) -> None:
    """
    Ejecuta inferencia sobre imagesTs/ y guarda predicciones en output_folder.
    Las predicciones se generan como archivos .nii.gz por imagen.
    """
    os.makedirs(output_folder, exist_ok=True)
    _run_cmd([
        'nnUNetv2_predict',
        '-i', input_folder,
        '-o', output_folder,
        '-d', str(dataset_id),
        '-c', config,
        '-f', str(fold),
    ])


def export_best_checkpoint(results_path: str, output_path: str,
                           dataset_id: int = DATASET_ID,
                           config: str = NNUNET_CONFIG,
                           fold: int = FOLD) -> None:
    """
    Copia el mejor checkpoint generado por nnU-Net al output_path.
    nnU-Net guarda el mejor modelo en:
      results_path/Dataset{id:03d}_Spine/{trainer}__{plan}__{config}/fold_{fold}/checkpoint_best.pth
    """
    dataset_folder = f'Dataset{dataset_id:03d}_{DATASET_NAME}'
    # nnU-Net v2 usa nnUNetTrainer__nnUNetPlans__{config} como nombre de experimento
    experiment     = f'nnUNetTrainer__nnUNetPlans__{config}'
    ckpt_src = os.path.join(
        results_path, dataset_folder, experiment, f'fold_{fold}', 'checkpoint_best.pth'
    )
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    shutil.copy2(ckpt_src, output_path)
    print(f'Checkpoint copiado a: {output_path}')

---
## Sección 3 — Métricas

nnU-Net produce segmentación semántica. No aplica AP@50 ni mAP.

### Carga de predicciones

In [ ]:
def load_nifti_prediction(pred_path: str) -> np.ndarray:
    """Carga máscara predicha por nnU-Net desde NIfTI. Retorna array uint8 (H,W)."""
    return nib.load(pred_path).get_fdata().astype(np.uint8)


def load_nifti_ground_truth(gt_path: str) -> np.ndarray:
    """Carga máscara ground truth desde NIfTI. Retorna array uint8 (H,W)."""
    return nib.load(gt_path).get_fdata().astype(np.uint8)


def collect_predictions_and_targets(test_df: pd.DataFrame,
                                    predictions_folder: str,
                                    dataset_root: str = DATASET_ROOT) -> tuple:
    """
    Itera sobre el test set y carga los pares (predicción, GT).
    La predicción viene de predictions_folder/{case_id}.nii.gz.
    El GT se reprocesa desde el dataset original (grayscale + map_entity_ids + ROI crop)
    para estar en el mismo espacio espacial que la entrada a nnU-Net.
    Si hay diferencia de tamaño se ajusta el GT con nearest neighbor.
    """
    predictions, targets = [], []

    for _, row in test_df.iterrows():
        case_id   = os.path.splitext(row['image'])[0]
        pred_path = os.path.join(predictions_folder, f'{case_id}.nii.gz')

        pred_mask = load_nifti_prediction(pred_path)

        # Reproducir el mismo preprocesamiento aplicado antes de convertir a nnU-Net
        image       = load_image(os.path.join(dataset_root, row['radiograph_path']))
        mask        = load_mask(os.path.join(dataset_root, row['multiclass_id_png']))
        binary_mask = load_binary_mask(os.path.join(dataset_root, row['label_binary_path']))
        mask_mapped = map_entity_ids(mask)
        roi         = compute_roi(binary_mask)
        _, gt_mask  = crop_to_roi(to_grayscale(image), mask_mapped, roi)

        # Ajustar tamaño si nnU-Net modificó la resolución internamente
        if pred_mask.shape != gt_mask.shape:
            gt_mask = cv2.resize(
                gt_mask,
                (pred_mask.shape[1], pred_mask.shape[0]),
                interpolation=cv2.INTER_NEAREST
            )

        predictions.append(pred_mask)
        targets.append(gt_mask)

    return predictions, targets

### Métricas de pixel (Dice e IoU)

In [ ]:
def compute_dice(pred: np.ndarray, gt: np.ndarray) -> float:
    """Dice entre dos máscaras binarias de la misma clase."""
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = (pred & gt).sum()
    denom = pred.sum() + gt.sum()
    return 2.0 * inter / denom if denom > 0 else 1.0


def compute_iou(pred: np.ndarray, gt: np.ndarray) -> float:
    """IoU entre dos máscaras binarias de la misma clase."""
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = (pred & gt).sum()
    union = (pred | gt).sum()
    return inter / union if union > 0 else 1.0


def compute_dice_per_class(predictions: list, targets: list,
                           num_classes: int = NUM_CLASSES) -> dict:
    """
    Dice por clase sobre el test set completo.
    Excluye clases ausentes en cada imagen (columna parcial).
    predictions[i] y targets[i] son máscaras semánticas (H,W) con IDs 0–22.
    """
    scores = {c: [] for c in range(1, num_classes)}
    for pred_mask, gt_mask in zip(predictions, targets):
        present = np.unique(gt_mask)
        present = present[present > 0]
        for c in present:
            gt_binary   = (gt_mask   == c).astype(np.uint8)
            pred_binary = (pred_mask == c).astype(np.uint8)
            scores[c].append(compute_dice(pred_binary, gt_binary))
    return {c: float(np.mean(v)) for c, v in scores.items() if v}


def compute_mean_dice(dice_per_class: dict) -> float:
    """Promedio de Dice sobre todas las clases presentes en el test set."""
    return float(np.mean(list(dice_per_class.values()))) if dice_per_class else 0.0


def compute_iou_per_class(predictions: list, targets: list,
                          num_classes: int = NUM_CLASSES) -> dict:
    """IoU por clase sobre el test set. Misma lógica que compute_dice_per_class."""
    scores = {c: [] for c in range(1, num_classes)}
    for pred_mask, gt_mask in zip(predictions, targets):
        present = np.unique(gt_mask)
        present = present[present > 0]
        for c in present:
            gt_binary   = (gt_mask   == c).astype(np.uint8)
            pred_binary = (pred_mask == c).astype(np.uint8)
            scores[c].append(compute_iou(pred_binary, gt_binary))
    return {c: float(np.mean(v)) for c, v in scores.items() if v}


def compute_miou(predictions: list, targets: list,
                 num_classes: int = NUM_CLASSES) -> float:
    """mIoU global sobre clases presentes en el test set."""
    iou_per_class = compute_iou_per_class(predictions, targets, num_classes)
    return float(np.mean(list(iou_per_class.values()))) if iou_per_class else 0.0

### Evaluación completa

In [ ]:
def evaluate_model(predictions: list, targets: list,
                   num_classes: int = NUM_CLASSES) -> dict:
    """
    Calcula métricas obligatorias para nnU-Net.
    AP@50 y mAP no aplican (segmentación semántica, no por instancia).
    """
    dice_per_class = compute_dice_per_class(predictions, targets, num_classes)
    iou_per_class  = compute_iou_per_class(predictions, targets, num_classes)
    return {
        'dice_per_class': dice_per_class,
        'iou_per_class':  iou_per_class,
        'mean_dice':      compute_mean_dice(dice_per_class),
        'miou':           compute_miou(predictions, targets, num_classes),
    }


def print_metrics_report(metrics: dict) -> None:
    """Tabla resumen de métricas por clase y globales."""
    header = f"{'Clase':<8} {'Dice':>8} {'IoU':>8}"
    print(header)
    print('-' * len(header))
    for c in range(1, NUM_CLASSES):
        name = CLASS_NAMES.get(c, f'ID{c}')
        dice = metrics['dice_per_class'].get(c, float('nan'))
        iou  = metrics['iou_per_class'].get(c, float('nan'))
        print(f"{name:<8} {dice:>8.4f} {iou:>8.4f}")
    print('-' * len(header))
    print(f"{'mean':<8} {metrics['mean_dice']:>8.4f} {metrics['miou']:>8.4f}")
    print()
    print(f"mean Dice: {metrics['mean_dice']:.4f}")
    print(f"mIoU:      {metrics['miou']:.4f}")
    print('\n(AP@50 / mAP no aplican a nnU-Net — segmentación semántica)')

---
## Pipeline

In [ ]:
# === PREPROCESAMIENTO — Conversión al formato nnU-Net ===
set_nnunet_env_vars(NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS)

index_df                  = load_dataset_index(DATASET_INDEX)
train_df, val_df, test_df = split_dataset(index_df, seed=SEED)
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

paths = create_nnunet_folder_structure(NNUNET_RAW, DATASET_ID, DATASET_NAME)
convert_dataset(train_df, val_df, test_df, paths)
generate_dataset_json(
    output_path=os.path.join(paths['root'], 'dataset.json'),
    num_training=len(train_df) + len(val_df),
)

# === PROCESAMIENTO — Entrenamiento nnU-Net ===
run_fingerprint_extraction(DATASET_ID)
run_plan_and_preprocess(DATASET_ID, NNUNET_CONFIG)
run_training(DATASET_ID, NNUNET_CONFIG, FOLD)
run_predict(
    input_folder=paths['imagesTs'],
    output_folder=PREDICTIONS_DIR,
    dataset_id=DATASET_ID,
    config=NNUNET_CONFIG,
    fold=FOLD,
)

# === GUARDAR MODELO FINAL ===
export_best_checkpoint(
    results_path=NNUNET_RESULTS,
    output_path=os.path.join(MODELS_DIR, 'nnunet_best.pth'),
    dataset_id=DATASET_ID,
    config=NNUNET_CONFIG,
    fold=FOLD,
)

# === MÉTRICAS ===
predictions, targets = collect_predictions_and_targets(test_df, PREDICTIONS_DIR)
metrics              = evaluate_model(predictions, targets)
print_metrics_report(metrics)